# T6: Data analysis in a streaming manner 

## Prerequisites

Kafka docker should be running. Required libraries need to be installed inside the container.

- docker-compose file for kafka
- `docker compose up -d`
- `docker exec -u 0 -it broker1-kr bash`
- `microdnf install python3 python3-pip -y`
- `pip3 install confluent-kafka pandas pyarrow faust`

In [ ]:
from confluent_kafka import Producer, Consumer, TopicPartition
import pyarrow.parquet as pq 
import pyarrow.compute as pc

import json, time, threading, heapq
from collections import deque
from datetime import datetime
from pathlib import Path 

import duckdb
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

# KAFKA SETTINGS
BOOTSTRAP_SERVERS = "localhost:10000,localhost:10001"
PRODUCER_SLEEP_DELAY = 0.01
KAFKA_PARTITION = 0

TOPIC_YELLOW = "taxi-yellow"
TOPIC_FHVHV = "taxi-fhvhv"

## Datasets as streams

The data selected for streaming: NYC YELLOW and FHVHV taxi data for year 2021. This data is divided by the source dataset (seperate files for yellow and fhvhv data), and records for the year are divided into multiple parquet files.


In [16]:
YELLOW_PATH = f"./data/yellow/year=2021/"
FHVHV_PATH = f"./data/fhvhv/year=2021/"
PICKUP_COL = "Pickup_DateTime"

YELLOW_COLUMNS = [
    'VendorID', 'Pickup_DateTime', 'Dropoff_DateTime', 'Passenger_Count',
    'Trip_Distance', 'Rate_Code', 'Store_And_Fwd_Flag', 'PULocationID',
    'DOLocationID', 'Pickup_Lon', 'Pickup_Lat', 'Dropoff_Lon',
    'Dropoff_Lat', 'Payment_Type', 'Fare_Amount', 'Surcharge', 'MTA_Tax',
    'Tip_Amount', 'Tolls_Amount', 'Total_Amount', 'Improvement_Surcharge',
    'Congestion_Surcharge', 'Airport_Fee', 'CBD_Congestion_Fee', 'year',]
 
FHVHV_COLUMNS = [
    'VendorID', 'Pickup_DateTime', 'Dropoff_DateTime', 'Store_And_Fwd_Flag',
    'Rate_Code', 'PULocationID', 'DOLocationID', 'Pickup_Lon', 'Pickup_Lat',
    'Dropoff_Lon', 'Dropoff_Lat', 'Passenger_Count', 'Trip_Distance',
    'Fare_Amount', 'Surcharge', 'MTA_Tax', 'Tip_Amount', 'Tolls_Amount',
    'Ehail_Fee', 'Improvement_Surcharge', 'Total_Amount', 'Payment_Type',
    'Trip_Type', 'Congestion_Surcharge', 'CBD_Congestion_Fee', 'year',]

STREAM_COLUMNS = ["Pickup_DateTime", "PULocationID", "DOLocationID", "Fare_Amount", "Trip_Distance", "Total_Amount"]
SELECTED_FEATURES = ["Fare_Amount", "Trip_Distance", "Total_Amount", "Passenger_Count"]

def list_files_for_year(files_dir):
    files_dir = Path(files_dir)
    files = sorted(files_dir.glob("*.parquet"))
    print("[INFO] Found", len(files), "parquet files for year 2021.")
    if not files:
        raise FileNotFoundError(f"No parquet files found in {files_dir}")
    return files 

### Top locations 

Find the overall top 10 locations. The top locations were selected as the locations with the highest amount of pickups and dropoffs overall, assuming that overall refers the entire dataset(s) for yellow and fhvhv taxis for year 2021.

In [17]:
def get_location_counts(path_to_dataset):
    return duckdb.query(f"""
    SELECT PULocationID as location_id, COUNT(*) AS location_count,
    FROM read_parquet('{path_to_dataset}') 
    WHERE YEAR(Pickup_DateTime) = 2021
    GROUP BY PULocationID
    ORDER BY PULocationID""").to_df()

location_counts_yellow = get_location_counts(YELLOW_PATH)
location_counts_fhvhv = get_location_counts(FHVHV_PATH)
location_counts = pd.concat([location_counts_yellow, location_counts_fhvhv])
location_counts = location_counts.groupby("location_id", as_index=False)["location_count"].sum().sort_values("location_count", ascending=False).head(10)
print("Number of pickups per location:")
print(location_counts)

TOP_10_LOCATIONS = list(location_counts["location_id"].values)
print("\nTop 10 locations based on the number of pickups:", (", ").join([str(l) for l in TOP_10_LOCATIONS]))

Number of pickups per location:
     location_id  location_count
78            79         3244263
129          132         3133946
234          237         2942310
60            61         2694919
233          236         2694212
135          138         2687498
158          161         2667114
231          234         2602508
47            48         2561500
167          170         2558966

Top 10 locations based on the number of pickups: 79, 132, 237, 61, 236, 138, 161, 234, 48, 170


### Combined chronological stream of data 

Each of the parquet files is first sorted by pickup timestamp. Instead of big, memory-consuming order by, the rows are from both datasets are combined into a chronologically ordered stream by k-way merge, and then rows get produced by Kafka Producer. 

In [18]:
PARQUET_BATCH_SIZE = 10000
BATCH_SIZE = 64

# ITERATATE OVER FILE FROM A SPECIFIC DATASET
def stream_file(path, cols, dataset):
    parquet_file = pq.ParquetFile(path)
    available_cols = [c for c in cols if c in parquet_file.schema_arrow.names]

    # SORT
    table = parquet_file.read(columns=available_cols) 
    sort_idx = pc.sort_indices(table, sort_keys=[(PICKUP_COL, "ascending")])
    table = table.take(sort_idx)

    # SEND BATCHES SO THAT IT IS MORE EFFICIENT 
    for batch in table.to_batches(max_chunksize=BATCH_SIZE):
        for row in batch.to_pylist():
            pickup_time = row.get(PICKUP_COL)
            if pickup_time is None:
                continue 
            msg = {"Dataset": dataset, **row}
            yield pickup_time, msg

# STREAM  FOR ONE SOURCE 
def source_stream(root, cols, dataset):
    files = list_files_for_year(root)
    for path in files:
        yield from stream_file(path=path, cols=cols, dataset=dataset)

# COMBINING BOTH YELLOW AND FHVHV DATA 
def combined_stream():
    yellow = source_stream(root=YELLOW_PATH, cols=YELLOW_COLUMNS, dataset="yellow")
    fhvhv = source_stream(root=FHVHV_PATH, cols=FHVHV_COLUMNS, dataset="fhvhv")
    # k-way merge sort 
    yield from heapq.merge(yellow, fhvhv, key=lambda x:x[0])

## Consumer

Consumer runs in a thread, so that the execution isn't blocked. Auto offset reset defines that all the messages will be read, not just the newest. Consumer subsribes to the Kafka topic and when it consumes messages, it decodes and displays received json. 

The rolling window is represented by a double-ended queue the size of the rolling window size. When a new message arrives, it gets added to the rolling window and the oldest message gets automatically removed from the rolling window queue. Rolling descriptive statistics are calculated by aggregation over locations (only for top 10 locations of interest), and boroughs (location IDs are mapped to boroughs using zone lookup). As the instructions did not specify if we should calculate statistics for pickup or dropoff locations/boroughs, I picked pickup ones to be aggregated over. Mean, sum, std, min and max values are calculated using pandas in-built functions. 

Unlike traditional clustering, where the cluster centroid are calculated over the entire dataset, when performing data anaylsis in a streaming manner, MiniBatchKMeans processes mini batches and updates existing cluster centers using the new observations and the model evolves over time. Mini batches fill cluster_data_batch until CLUSTER_MINIBATCH_SIZE samples are observed and then scales the numeric data, and performes the partial fit. 

In [19]:
zone_lookup = pd.read_csv("./data/taxi_zone_lookup.csv")
zone_to_borough = zone_lookup.set_index("LocationID")["Borough"].to_dict()

In [20]:
# ROLLING DESC STATS  ================================
WINDOW_SIZE = 100 
rolling_window_borough = deque(maxlen=WINDOW_SIZE)
rolling_window_locations = deque(maxlen=WINDOW_SIZE)

def calc_rolling_stats_borough(rows):
    df = pd.DataFrame(rows)
    print("\nROLLING STATS FOR BOROUGHS:")
    df["PUBorough"] = df["PULocationID"].map(zone_to_borough)
    stats_borough = df.groupby('PUBorough')[SELECTED_FEATURES].agg(['mean', 'sum', 'std', 'min', 'max'])
    print(stats_borough)

def calc_rolling_stats(rows):
    df = pd.DataFrame(rows)
    loc_stats = df.groupby('PULocationID')[SELECTED_FEATURES].agg(['mean', 'sum', 'std', 'min', 'max'])
    print("\nROLLING STATS FOR TOP 10 LOCATIONS:")
    print(loc_stats)

# CLUSTERING ============================
N_CLUSTERS = 5 
CLUSTER_MINIBATCH_SIZE = 100
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=0, batch_size=CLUSTER_MINIBATCH_SIZE, n_init="auto")
scaler = StandardScaler()

# CONSUMER ===============================
consumer = Consumer({
    'bootstrap.servers': BOOTSTRAP_SERVERS,
    "group.id": "taxi-consumer",
    'auto.offset.reset': 'earliest'})
consumer.subscribe([TOPIC_YELLOW, TOPIC_FHVHV])
consumer_is_running = True

def run_consumer():
    cluster_data_batch = []
    while consumer_is_running:
        message = consumer.poll(1.0)
        if message is None:
            continue
        if message.error():
            print(message.error())
            continue

        consumed_data = json.loads(message.value().decode('utf-8'))
        print("Message received from topic:", message.topic())
        print(consumed_data)

        has_all_features = all(consumed_data.get(key) is not None for key in SELECTED_FEATURES)
        #print(consumed_data)

        if has_all_features:
            # ROLLING DESC STATS  ==========================
            rolling_window_borough.append(consumed_data)
            if len(rolling_window_borough)>=WINDOW_SIZE:
                calc_rolling_stats_borough(list(rolling_window_borough))

            if (int(consumed_data["PULocationID"]) in TOP_10_LOCATIONS):
                rolling_window_locations.append(consumed_data)
                if len(rolling_window_locations)>=WINDOW_SIZE:
                    calc_rolling_stats(list(rolling_window_locations))

            # CLUSTERING ===================================
            cluster_data_batch.append(consumed_data)

        if len(cluster_data_batch)==CLUSTER_MINIBATCH_SIZE:
            X_batch = pd.DataFrame(cluster_data_batch)[SELECTED_FEATURES]
            X_batch = X_batch.replace([np.inf, -np.inf], np.nan)
            X_batch = X_batch.dropna()
            if X_batch.empty:
                cluster_data_batch = []
                continue 
            
            cluster_data_batch = []
            scaler.partial_fit(X_batch)
            X_batch_scaled = scaler.transform(X_batch)
            kmeans.partial_fit(X_batch_scaled)
            cluster_label = kmeans.predict(X_batch_scaled)
            print("=== CLUSTERS ===")
            print("\nClustering labels:")
            print(cluster_label[:10])

consumer_thread = threading.Thread(target=run_consumer, daemon=True)
consumer_thread.start()

## Producer 

First, producer is configured. Then, data is converted to a more efficient format for streaming. For each row, message is constructed and converted into jason, which is then handled by the producer.produce(). Flush producer.flush() is called at the end, after all the messages has been sent. 

In [21]:
print(f"[INFO] Producer starting.")
producer = Producer({'bootstrap.servers': BOOTSTRAP_SERVERS})
nr_sent = 0
 
def convert_to_iso(obj):
    if isinstance(obj, datetime):
            return obj.isoformat()
try:
    for pickup_time, msg in combined_stream():
        topic = ""
        if msg["Dataset"]=="yellow":
            topic = TOPIC_YELLOW
        elif msg["Dataset"]=="fhvhv":
            topic = TOPIC_FHVHV
        else:
            print("Unknown source.")
            continue
             
        message_json = json.dumps(msg, default=convert_to_iso).encode('utf-8')
        producer.produce(topic, partition=KAFKA_PARTITION, value=message_json)
        producer.poll(0)
        nr_sent+=1 
        time.sleep(PRODUCER_SLEEP_DELAY)
except Exception as e:
    print("[EXCEPTION]", e)

producer.flush()
print(f"[INFO] Streaming finished. Steamed {nr_sent} rows.")


[INFO] Producer starting.
[INFO] Found 12 parquet files for year 2021.
[INFO] Found 12 parquet files for year 2021.


KeyboardInterrupt: 

## Stopping consumer 

Since consumer is always running and waiting to consume messages, it must be stopped manually by setting consumer_is_running to false.

In [ ]:
# STOP CONSUMER
consumer_is_running = False
consumer.close()

: 